In [9]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
words = open('/content/drive/MyDrive/Colab Notebooks/names.txt', 'r').read().splitlines()

words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words)

32033

In [6]:
# build vocab of chars from s to i and i to s

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [78]:
# building the dataset

block_size = 3
X, Y = [], []
context = [0] * block_size

for w in words:

  for ch in w + '.':
    idx = stoi[ch]
    X.append(context)
    Y.append(idx)
    context = context[1:] + [idx]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [79]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [115]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [134]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2),generator=g)

W1 = torch.randn((6, 100),generator=g)
b1 = torch.randn(100,generator=g)

W2 = torch.randn((100, 27),generator=g)
b2 = torch.randn(27,generator=g)

parameters = [C, W1, b1, W2, b2]

In [135]:
sum(p.nelement() for p in parameters)

3481

In [136]:
for p in parameters:
  p.requires_grad = True

In [137]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [146]:
lri = []
lossi = []

for i in range(10000):
  # mini-batch construct
  ix = torch.randint(0, X.shape[0], (32,))

  # forward pass
  emb = C[X[ix]] # (32, 3, 2)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # emb @ W1 == (32, 100) + b == (100) so broadcasting is good.
  logits = h @ W2 + b2 # (32, 27)
  # counts = logits.exp()
  # probs = counts / counts.sum(1, keepdims=True)
  # loss = -probs[torch.arange(32), Y].log().mean()
  # easy alternative is:
  loss = F.cross_entropy(logits, Y[ix])
  # print(loss.item())

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  # lr = lrs[i]
  lr = 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # stats
  # lri.append(lre[i])
  # lossi.append(loss.item())

print(loss.item())

2.3424854278564453


In [ ]:
plt.plot(lri, lossi)

In [147]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y)
loss

tensor(2.3376, grad_fn=<NllLossBackward0>)